In [2]:
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np
import sklearn
from sklearn.model_selection import GridSearchCV 

In [3]:
df = pd.read_csv("D:\\sir files\\Data science and machien learning\\case study\\PimaIndiansDiabetes.csv")

In [4]:
df

,TimesPregnant,GlucoseConcentration,BloodPrs,SkinThickness,Serum,BMI,DiabetesFunct,Age,Class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [5]:
df.isna().sum()

TimesPregnant           0
GlucoseConcentration    0
BloodPrs                0
SkinThickness           0
Serum                   0
BMI                     0
DiabetesFunct           0
Age                     0
Class                   0
dtype: int64

In [6]:
df.columns

Index(['TimesPregnant', 'GlucoseConcentration', 'BloodPrs', 'SkinThickness',
       'Serum', 'BMI', 'DiabetesFunct', 'Age', 'Class'],
      dtype='object')

In [7]:
df.shape

(768, 9)

In [8]:
df.size

6912

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   TimesPregnant         768 non-null    int64  
 1   GlucoseConcentration  768 non-null    int64  
 2   BloodPrs              768 non-null    int64  
 3   SkinThickness         768 non-null    int64  
 4   Serum                 768 non-null    int64  
 5   BMI                   768 non-null    float64
 6   DiabetesFunct         768 non-null    float64
 7   Age                   768 non-null    int64  
 8   Class                 768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [10]:
df.describe()

,TimesPregnant,GlucoseConcentration,BloodPrs,SkinThickness,Serum,BMI,DiabetesFunct,Age,Class
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [11]:
df.Class.nunique()

2

In [12]:
df["Class"].unique()

array([1, 0])

In [13]:
df.head(5)

,TimesPregnant,GlucoseConcentration,BloodPrs,SkinThickness,Serum,BMI,DiabetesFunct,Age,Class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:


python -m venv venv
source venv/bin/activate
pip install pandas numpy scikit-learn xgboost joblib fastapi uvicorn pydantic shap






# train_and_save.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
import joblib
import json
import os

# 1) Load data
df = pd.read_csv("data.csv")
TARGET = "target"

# quick checks
print("shape:", df.shape)
print(df[TARGET].value_counts(normalize=True))

# 2) Train / holdout split
train_df, holdout_df = train_test_split(df, test_size=0.15, stratify=df[TARGET], random_state=42)
X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]
X_hold = holdout_df.drop(columns=[TARGET])
y_hold = holdout_df[TARGET]

# 3) Identify column types (simple heuristic)
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# (Optional) drop or transform high-cardinality or ID columns
# Example: if 'id' present:
if 'id' in numeric_cols:
    numeric_cols.remove('id')

# 4) Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
], sparse_threshold=0)

# 5) Modeling pipeline (XGBoost as strong model)
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    n_jobs=4
)

pipeline = Pipeline(steps=[("preprocessor", preprocessor),
                           ("model", model)])

# 6) Quick cross-validated baseline (optional: use StratifiedKFold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Quick default CV score
from sklearn.model_selection import cross_val_score
cv_score = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=4)
print("CV ROC AUC (mean):", cv_score.mean())

# 7) Hyperparameter tuning (RandomizedSearchCV example)
param_dist = {
    "model__n_estimators": [100, 200, 400],
    "model__max_depth": [3, 5, 8],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=4
)

search.fit(X, y)
print("Best CV ROC AUC:", search.best_score_)
print("Best params:", search.best_params_)

best_pipeline = search.best_estimator_

# 8) Evaluate on holdout
y_pred_proba = best_pipeline.predict_proba(X_hold)[:, 1]
y_pred = best_pipeline.predict(X_hold)
print("Holdout ROC AUC:", roc_auc_score(y_hold, y_pred_proba))
print(classification_report(y_hold, y_pred))

# 9) Save pipeline (preprocessor + model)
os.makedirs("models", exist_ok=True)
joblib.dump(best_pipeline, "models/pipeline_v1.joblib")
# Save metadata
meta = dict(columns=list(X.columns), numeric_cols=numeric_cols, categorical_cols=categorical_cols)
with open("models/pipeline_v1_meta.json", "w") as f:
    json.dump(meta, f)

print("Saved pipeline to models/pipeline_v1.joblib")









import shap
best = best_pipeline.named_steps['model']
# get transformed feature names (if you need mapping)
X_transformed = best_pipeline.named_steps['preprocessor'].transform(X)
explainer = shap.TreeExplainer(best)
shap_values = explainer.shap_values(X_transformed)
shap.summary_plot(shap_values, X_transformed, feature_names=...)  # careful to pass correct feature names







# app/main.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd

app = FastAPI(title="Model API")

class InputRow(BaseModel):
    # If you have many columns, you can accept dicts instead:
    data: dict

MODEL_PATH = "/app/models/pipeline_v1.joblib"
model = joblib.load(MODEL_PATH)

@app.get("/")
def root():
    return {"status": "ok", "model": "pipeline_v1"}

@app.post("/predict")
def predict(payload: InputRow):
    # payload.data should be a mapping: {"col1": value, "col2": value, ...}
    df = pd.DataFrame([payload.data])
    preds_proba = model.predict_proba(df)[:, 1]
    preds = model.predict(df)
    return {"prediction": int(preds[0]), "probability": float(preds_proba[0])}







FROM python:3.11-slim

WORKDIR /app

# Install system deps if needed (e.g., build-essential) - keep minimal for prod images
RUN apt-get update && apt-get install -y --no-install-recommends \
    gcc \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app /app
COPY models /app/models

EXPOSE 8000

# Use Uvicorn with multiple workers (adjust as needed)
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]












docker build -t mymodel:latest .
docker run -p 8000:8000 mymodel:latest
# Test:
curl -X POST "http://localhost:8000/predict" -H "Content-Type: application/json" \
  -d '{"data": {"col1": 1, "col2": "abc", ...}}'


In [ ]:




# End-to-End ML repo (Customized for your CSV)

This document contains a ready GitHub-style repository layout customized for the uploaded dataset `premier_league_table_13.csv` (season snapshot after 13 games). It includes training scripts to build multiple models you requested: predict **Points**, **Goals For / Goals Against / Goal Difference**, **league-ranking classifications** (Top-4 binary and 3-class tiers), and a plan/placeholders for **next-match win probability** (requires match-level data).

---

## repo structure

```
e2e-ml-repo/
├── README.md
├── requirements.txt
├── .gitignore
├── train_and_save_multi.py     # trains multiple targets and saves models
├── utils.py
├── app/
│   ├── main.py                # API serving multiple models
│   └── schemas.py
├── models/                    # output - saved model files
│   └── (generated by training)
├── Dockerfile
└── examples/
    ├── predict_points.json
    ├── predict_goals.json
    └── predict_rank.json
```

---

## README.md (summary)

````markdown
# E2E ML Repo - Premier League Table (customized)

This repo is customized for `premier_league_table_13.csv` (team-level season summary after 13 matches). It trains multiple models:

- Regression: predict `Points`, `Goals For`, `Goals Against`, `Goal Difference`
- Classification: Top-4 (binary), 3-tier rank (Top / Mid / Bottom)
- Placeholder & plan for next-match win probability (requires match-level dataset)

Run training:

```bash
python train_and_save_multi.py --data premier_league_table_13.csv --team_col Team
````

This will save model files under `models/`:

* `points_reg.joblib`
* `goals_for_reg.joblib`
* `goals_against_reg.joblib`
* `gd_reg.joblib`
* `top4_clf.joblib`
* `tier3_clf.joblib`

Serve models with FastAPI:

```bash
# build docker (if desired)
docker build -t e2e-ml-custom:latest .
# or run locally
uvicorn app.main:app --reload --host 0.0.0.0 --port 8000
```

Test example request files are in `examples/`.

```

---

## requirements.txt

```

fastapi
uvicorn[standard]
pandas
numpy
scikit-learn
xgboost
joblib
pydantic
shap

```

---

## .gitignore

```

venv/
**pycache**/
models/
*.pyc
.env
.DS_Store

````

---

## train_and_save_multi.py

```python
"""Train multiple models (regression and classification) for the provided aggregated table.
Usage: python train_and_save_multi.py --data premier_league_table_13.csv --team_col Team
"""
import argparse
import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, classification_report
from utils import infer_column_types

SEED = 42


def make_features(df):
    df = df.copy()
    # Basic engineered features
    df['win_pct'] = df['Won'] / df['Played']
    df['loss_pct'] = df['Lost'] / df['Played']
    df['draw_pct'] = df['Drawn'] / df['Played']
    df['goals_for_per_game'] = df['Goals For'] / df['Played']
    df['goals_against_per_game'] = df['Goals Against'] / df['Played']
    # defensive/offensive ratio
    df['gf_ga_ratio'] = df['Goals For'] / (df['Goals Against'].replace(0, np.nan))
    df['goal_diff_per_game'] = df['Goal Difference'] / df['Played']
    # fill inf/nan
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    return df


def build_and_save_regression(X, y, model_path):
    num_cols, cat_cols = infer_column_types(X)
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False)),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ], sparse_threshold=0)

    model = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=SEED)
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])

    pipeline.fit(X, y)
    preds = pipeline.predict(X)
    print(f"Trained reg model {model_path} - R2: {r2_score(y, preds):.4f}, RMSE: {mean_squared_error(y, preds, squared=False):.4f}")

    joblib.dump(pipeline, model_path)


def build_and_save_classifier(X, y, model_path):
    num_cols, cat_cols = infer_column_types(X)
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False)),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ], sparse_threshold=0)

    model = XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='logloss', random_state=SEED)
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])

    pipeline.fit(X, y)
    preds = pipeline.predict(X)
    print(f"Trained clf model {model_path} - sample classification report:
", classification_report(y, preds))

    joblib.dump(pipeline, model_path)


def create_rank_tiers(df, top_n=4):
    # Create Top-4 binary and 3-tier classification
    df = df.copy().reset_index(drop=True)
    df = df.sort_values('Points', ascending=False).reset_index(drop=True)
    df['rank'] = df.index + 1
    df['top4'] = (df['rank'] <= top_n).astype(int)
    # 3 tiers: Top (rank <= top_n), Mid (top_n<rank<= top_n+6), Bottom else
    df['tier3'] = 1  # default Mid
    df.loc[df['rank'] <= top_n, 'tier3'] = 0  # Top
    df.loc[df['rank'] > top_n+6, 'tier3'] = 2  # Bottom
    return df


def main(args):
    df = pd.read_csv(args.data)
    print('Loaded:', df.shape)
    df = make_features(df)

    # Features: drop identifiers and target columns when forming X
    feature_cols = [c for c in df.columns if c not in ['Team', 'Points', 'Goals For', 'Goals Against', 'Goal Difference', 'rank', 'top4', 'tier3']]
    X = df[feature_cols]

    os.makedirs('models', exist_ok=True)

    # 1) Points regression
    y_points = df['Points']
    build_and_save_regression(X, y_points, 'models/points_reg.joblib')

    # 2) Goals For regression
    y_gf = df['Goals For']
    build_and_save_regression(X, y_gf, 'models/goals_for_reg.joblib')

    # 3) Goals Against regression
    y_ga = df['Goals Against']
    build_and_save_regression(X, y_ga, 'models/goals_against_reg.joblib')

    # 4) Goal Difference regression
    y_gd = df['Goal Difference']
    build_and_save_regression(X, y_gd, 'models/gd_reg.joblib')

    # 5) Ranking / Classification
    df_ranked = create_rank_tiers(df, top_n=4)
    # top4 classifier
    build_and_save_classifier(X.loc[df_ranked.index], df_ranked['top4'], 'models/top4_clf.joblib')
    # tier3 classifier
    build_and_save_classifier(X.loc[df_ranked.index], df_ranked['tier3'], 'models/tier3_clf.joblib')

    # Save metadata
    meta = {
        'feature_cols': feature_cols,
        'numeric_in_sample': X.select_dtypes(include=['number']).columns.tolist(),
        'categorical_in_sample': X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    }
    with open('models/meta.json', 'w') as f:
        json.dump(meta, f)

    print('All models saved to models/')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', required=True)
    parser.add_argument('--team_col', default='Team')
    args = parser.parse_args()
    main(args)
````

---

## utils.py

```python
import pandas as pd


def infer_column_types(df: pd.DataFrame):
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    # drop Team if present
    for c in ['Team', 'team', 'ID', 'Id', 'id']:
        if c in categorical_cols:
            categorical_cols.remove(c)
        if c in numeric_cols:
            numeric_cols.remove(c)
    return numeric_cols, categorical_cols
```

---

## app/schemas.py

```python
from pydantic import BaseModel
from typing import Dict, Any

class PredictRequest(BaseModel):
    data: Dict[str, Any]

class PredictResponse(BaseModel):
    predictions: Dict[str, float]
```

---

## app/main.py

```python
from fastapi import FastAPI, HTTPException
from app.schemas import PredictRequest, PredictResponse
import joblib
import pandas as pd
import os

app = FastAPI(title='E2E ML - Premier League models')

MODEL_DIR = os.environ.get('MODEL_DIR', '/app/models')
# Attempt to load models if present
models = {}
for name in ['points_reg', 'goals_for_reg', 'goals_against_reg', 'gd_reg', 'top4_clf', 'tier3_clf']:
    path = os.path.join(MODEL_DIR, f'{name}.joblib')
    if os.path.exists(path):
        try:
            models[name] = joblib.load(path)
        except Exception as e:
            print('Failed to load', path, e)

@app.get('/')
def root():
    return {'status':'ok', 'loaded_models': list(models.keys())}

@app.post('/predict', response_model=PredictResponse)
def predict(req: PredictRequest):
    if not models:
        raise HTTPException(status_code=503, detail='No models loaded')
    df = pd.DataFrame([req.data])
    out = {}
    for name, m in models.items():
        try:
            if hasattr(m, 'predict_proba') and name.startswith('top4') or name.startswith('tier3'):
                pred = m.predict(df)
                # for classifiers return predicted class
                out[name] = int(pred[0])
            else:
                pred = m.predict(df)
                out[name] = float(pred[0])
        except Exception as e:
            out[name] = None
    return {'predictions': out}
```

---

## examples/predict_points.json

```json
{
  "data": {
    "Played": 13,
    "Won": 9,
    "Drawn": 3,
    "Lost": 1,
    "Goals For": 25,
    "Goals Against": 7,
    "Goal Difference": 18
  }
}
```

---

## Notes about "Next-match win probability"

You asked for next-match win probability as well. **Important limitation:** the uploaded file is a season summary (one row per team) and does **not** contain match-by-match events or opponent pairings. Predicting a single-match win probability reliably requires match-level data (home/away, opponent, date, injuries, lineups, historical head-to-head, form).

What I included instead:

* A **plan** and placeholder code comments in `train_and_save_multi.py` describing how to train a match-level model if you provide a match dataset (columns: date, home_team, away_team, home_goals, away_goals, venue, etc.).
* A suggested simple approach (Poisson goal model / Elo + logistic regression) which I can implement as soon as you upload match-level data.

---

## How to run locally

1. Install requirements: `pip install -r requirements.txt`
2. Train: `python train_and_save_multi.py --data premier_league_table_13.csv`
3. Run API (local): `uvicorn app.main:app --reload --port 8000`
4. Test with one of the example JSONs in `examples/`.

---

If you'd like, I can now:

* (A) Produce the **trained models** by running the training here on your uploaded CSV and show evaluation metrics and artifacts (I can run training on the provided CSV *now*).
* (B) Instead, give you the files to run locally.

Pick A or B. If you pick A, I'll run training on the CSV you uploaded and return results (metrics, SHAP plots, and the saved model files).


In [ ]:
# End-to-End ML repo (Customized for your CSV)

This document contains a ready GitHub-style repository layout customized for the uploaded dataset `premier_league_table_13.csv` (season snapshot after 13 games). It includes training scripts to build multiple models you requested: predict **Points**, **Goals For / Goals Against / Goal Difference**, **league-ranking classifications** (Top-4 binary and 3-class tiers), and a plan/placeholders for **next-match win probability** (requires match-level data).

---

## repo structure

```
e2e-ml-repo/
├── README.md
├── requirements.txt
├── .gitignore
├── train_and_save_multi.py     # trains multiple targets and saves models
├── utils.py
├── app/
│   ├── main.py                # API serving multiple models
│   └── schemas.py
├── models/                    # output - saved model files
│   └── (generated by training)
├── Dockerfile
└── examples/
    ├── predict_points.json
    ├── predict_goals.json
    └── predict_rank.json
```

---

## README.md (summary)

````markdown
# E2E ML Repo - Premier League Table (customized)

This repo is customized for `premier_league_table_13.csv` (team-level season summary after 13 matches). It trains multiple models:

- Regression: predict `Points`, `Goals For`, `Goals Against`, `Goal Difference`
- Classification: Top-4 (binary), 3-tier rank (Top / Mid / Bottom)
- Placeholder & plan for next-match win probability (requires match-level dataset)

Run training:

```bash
python train_and_save_multi.py --data premier_league_table_13.csv --team_col Team
````

This will save model files under `models/`:

* `points_reg.joblib`
* `goals_for_reg.joblib`
* `goals_against_reg.joblib`
* `gd_reg.joblib`
* `top4_clf.joblib`
* `tier3_clf.joblib`

Serve models with FastAPI:

```bash
# build docker (if desired)
docker build -t e2e-ml-custom:latest .
# or run locally
uvicorn app.main:app --reload --host 0.0.0.0 --port 8000
```

Test example request files are in `examples/`.

```

---

## requirements.txt

```

fastapi
uvicorn[standard]
pandas
numpy
scikit-learn
xgboost
joblib
pydantic
shap

```

---

## .gitignore

```

venv/
**pycache**/
models/
*.pyc
.env
.DS_Store

````

---

## train_and_save_multi.py

```python
"""Train multiple models (regression and classification) for the provided aggregated table.
Usage: python train_and_save_multi.py --data premier_league_table_13.csv --team_col Team
"""
import argparse
import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import r2_score, mean_squared_error, roc_auc_score, classification_report
from utils import infer_column_types

SEED = 42


def make_features(df):
    df = df.copy()
    # Basic engineered features
    df['win_pct'] = df['Won'] / df['Played']
    df['loss_pct'] = df['Lost'] / df['Played']
    df['draw_pct'] = df['Drawn'] / df['Played']
    df['goals_for_per_game'] = df['Goals For'] / df['Played']
    df['goals_against_per_game'] = df['Goals Against'] / df['Played']
    # defensive/offensive ratio
    df['gf_ga_ratio'] = df['Goals For'] / (df['Goals Against'].replace(0, np.nan))
    df['goal_diff_per_game'] = df['Goal Difference'] / df['Played']
    # fill inf/nan
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
    return df


def build_and_save_regression(X, y, model_path):
    num_cols, cat_cols = infer_column_types(X)
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False)),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ], sparse_threshold=0)

    model = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=SEED)
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])

    pipeline.fit(X, y)
    preds = pipeline.predict(X)
    print(f"Trained reg model {model_path} - R2: {r2_score(y, preds):.4f}, RMSE: {mean_squared_error(y, preds, squared=False):.4f}")

    joblib.dump(pipeline, model_path)


def build_and_save_classifier(X, y, model_path):
    num_cols, cat_cols = infer_column_types(X)
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False)),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ], sparse_threshold=0)

    model = XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='logloss', random_state=SEED)
    pipeline = Pipeline([("preprocessor", preprocessor), ("model", model)])

    pipeline.fit(X, y)
    preds = pipeline.predict(X)
    print(f"Trained clf model {model_path} - sample classification report:
", classification_report(y, preds))

    joblib.dump(pipeline, model_path)


def create_rank_tiers(df, top_n=4):
    # Create Top-4 binary and 3-tier classification
    df = df.copy().reset_index(drop=True)
    df = df.sort_values('Points', ascending=False).reset_index(drop=True)
    df['rank'] = df.index + 1
    df['top4'] = (df['rank'] <= top_n).astype(int)
    # 3 tiers: Top (rank <= top_n), Mid (top_n<rank<= top_n+6), Bottom else
    df['tier3'] = 1  # default Mid
    df.loc[df['rank'] <= top_n, 'tier3'] = 0  # Top
    df.loc[df['rank'] > top_n+6, 'tier3'] = 2  # Bottom
    return df


def main(args):
    df = pd.read_csv(args.data)
    print('Loaded:', df.shape)
    df = make_features(df)

    # Features: drop identifiers and target columns when forming X
    feature_cols = [c for c in df.columns if c not in ['Team', 'Points', 'Goals For', 'Goals Against', 'Goal Difference', 'rank', 'top4', 'tier3']]
    X = df[feature_cols]

    os.makedirs('models', exist_ok=True)

    # 1) Points regression
    y_points = df['Points']
    build_and_save_regression(X, y_points, 'models/points_reg.joblib')

    # 2) Goals For regression
    y_gf = df['Goals For']
    build_and_save_regression(X, y_gf, 'models/goals_for_reg.joblib')

    # 3) Goals Against regression
    y_ga = df['Goals Against']
    build_and_save_regression(X, y_ga, 'models/goals_against_reg.joblib')

    # 4) Goal Difference regression
    y_gd = df['Goal Difference']
    build_and_save_regression(X, y_gd, 'models/gd_reg.joblib')

    # 5) Ranking / Classification
    df_ranked = create_rank_tiers(df, top_n=4)
    # top4 classifier
    build_and_save_classifier(X.loc[df_ranked.index], df_ranked['top4'], 'models/top4_clf.joblib')
    # tier3 classifier
    build_and_save_classifier(X.loc[df_ranked.index], df_ranked['tier3'], 'models/tier3_clf.joblib')

    # Save metadata
    meta = {
        'feature_cols': feature_cols,
        'numeric_in_sample': X.select_dtypes(include=['number']).columns.tolist(),
        'categorical_in_sample': X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    }
    with open('models/meta.json', 'w') as f:
        json.dump(meta, f)

    print('All models saved to models/')


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', required=True)
    parser.add_argument('--team_col', default='Team')
    args = parser.parse_args()
    main(args)
````

---

## utils.py

```python
import pandas as pd


def infer_column_types(df: pd.DataFrame):
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    # drop Team if present
    for c in ['Team', 'team', 'ID', 'Id', 'id']:
        if c in categorical_cols:
            categorical_cols.remove(c)
        if c in numeric_cols:
            numeric_cols.remove(c)
    return numeric_cols, categorical_cols
```

---

## app/schemas.py

```python
from pydantic import BaseModel
from typing import Dict, Any

class PredictRequest(BaseModel):
    data: Dict[str, Any]

class PredictResponse(BaseModel):
    predictions: Dict[str, float]
```

---

## app/main.py

```python
from fastapi import FastAPI, HTTPException
from app.schemas import PredictRequest, PredictResponse
import joblib
import pandas as pd
import os

app = FastAPI(title='E2E ML - Premier League models')

MODEL_DIR = os.environ.get('MODEL_DIR', '/app/models')
# Attempt to load models if present
models = {}
for name in ['points_reg', 'goals_for_reg', 'goals_against_reg', 'gd_reg', 'top4_clf', 'tier3_clf']:
    path = os.path.join(MODEL_DIR, f'{name}.joblib')
    if os.path.exists(path):
        try:
            models[name] = joblib.load(path)
        except Exception as e:
            print('Failed to load', path, e)

@app.get('/')
def root():
    return {'status':'ok', 'loaded_models': list(models.keys())}

@app.post('/predict', response_model=PredictResponse)
def predict(req: PredictRequest):
    if not models:
        raise HTTPException(status_code=503, detail='No models loaded')
    df = pd.DataFrame([req.data])
    out = {}
    for name, m in models.items():
        try:
            if hasattr(m, 'predict_proba') and name.startswith('top4') or name.startswith('tier3'):
                pred = m.predict(df)
                # for classifiers return predicted class
                out[name] = int(pred[0])
            else:
                pred = m.predict(df)
                out[name] = float(pred[0])
        except Exception as e:
            out[name] = None
    return {'predictions': out}
```

---

## examples/predict_points.json

```json
{
  "data": {
    "Played": 13,
    "Won": 9,
    "Drawn": 3,
    "Lost": 1,
    "Goals For": 25,
    "Goals Against": 7,
    "Goal Difference": 18
  }
}
```

---

## Notes about "Next-match win probability"

You asked for next-match win probability as well. **Important limitation:** the uploaded file is a season summary (one row per team) and does **not** contain match-by-match events or opponent pairings. Predicting a single-match win probability reliably requires match-level data (home/away, opponent, date, injuries, lineups, historical head-to-head, form).

What I included instead:

* A **plan** and placeholder code comments in `train_and_save_multi.py` describing how to train a match-level model if you provide a match dataset (columns: date, home_team, away_team, home_goals, away_goals, venue, etc.).
* A suggested simple approach (Poisson goal model / Elo + logistic regression) which I can implement as soon as you upload match-level data.

---

## How to run locally

1. Install requirements: `pip install -r requirements.txt`
2. Train: `python train_and_save_multi.py --data premier_league_table_13.csv`
3. Run API (local): `uvicorn app.main:app --reload --port 8000`
4. Test with one of the example JSONs in `examples/`.

---

If you'd like, I can now:

* (A) Produce the **trained models** by running the training here on your uploaded CSV and show evaluation metrics and artifacts (I can run training on the provided CSV *now*).
* (B) Instead, give you the files to run locally.

Pick A or B. If you pick A, I'll run training on the CSV you uploaded and return results (metrics, SHAP plots, and the saved model files).
